# RAG architecture: BM25 + Chroma + rerank
### Annual report question answering with Qwen, LangChain, Pydantic and grounded fallback

Hybrid search with Reciprocal Rank Fusion, then a cross-encoder reranker. This is the fallback if Qdrant isn't approved.

- Charts and images: can't see them.
- Tables and figures: good on readable text, because exact terms and meaning are both searched.
- Ingest cost: free. The reranker scores 40-60 pairs per question, so retrieval is slower.

**How it runs:** retrieve evidence, have Qwen return JSON that must validate against a Pydantic schema, then check the answer with deterministic gates. Settings, parsed pages, gold questions, results and metrics are Pydantic models too. If the LLM gives no verified answer, the notebook shows the ground truth and labels it as such.

**What it measures:** numeric number match rate, faithfulness (target > 0.98) and negative refusal accuracy (target > 0.95), scored on the LLM's own answers. A ground-truth fallback never counts as a correct LLM answer.

**To run:** choose a T4 GPU, then Runtime > Run all. Allow Google Drive access when asked, or untick `USE_DRIVE` in the run options to keep everything in `/content`, which is lost when the runtime ends. Upload the PDF when asked; with Drive you only do this once, because the other notebooks find it in the same folder. Tick `QUICK_TEST` for a short check before a full run. The gold questions were written for `EDGAR_ARS_CVX_2024.pdf`; any other PDF still works for your own questions.

## Step 1: Environment setup

In [ ]:
# Colab: Runtime > Change runtime type > T4 GPU first.
!nvidia-smi -L
!pip install -q langchain-core langchain-community langchain-huggingface langchain-text-splitters 'pydantic>=2' pymupdf rapidocr onnxruntime sentence-transformers accelerate bitsandbytes pandas rank_bm25 langchain-chroma chromadb

In [ ]:
# @title Run options
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_FOLDER = "rag_arch"  # @param {type:"string"}
QUICK_TEST = False  # @param {type:"boolean"}

# With Drive, the PDF, VLM cache, results and scoreboard survive a disconnect and are shared by all seven notebooks.
# A quick test evaluates 6 questions and sends 3 pages to the VLM: it checks the notebook runs, so don't report its scores.
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RAG_WORK_DIR"] = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
for name, value in {"EVAL_LIMIT": "6", "VLM_MAX_PAGES": "3"}.items():
    if QUICK_TEST:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)

## Step 2: Upload the PDF and set the configuration

In [ ]:
ARCHITECTURE = "03_bm25_chroma_rerank"
EXPECTED_PDF = "EDGAR_ARS_CVX_2024.pdf"
EXPECTED_SHA256 = "e3b035cfdf2f079ea812da5340a12bd4941229cb2ce4e12d949cbe869c0d9cb2"
import gc, glob, hashlib, json, os, re, shutil, statistics, time
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Annotated, Literal, Optional, Union

import pandas as pd
import torch
from pydantic import BaseModel, ConfigDict, Field, TypeAdapter, ValidationError, computed_field, field_validator, model_validator

class Settings(BaseModel):
    """Run settings, validated once. Environment variables override the defaults."""
    model_config = ConfigDict(frozen=True)
    work_dir: Path = Path("/content")
    qwen_model: str = "Qwen/Qwen2.5-7B-Instruct"  # "Qwen/Qwen2.5-8B-Instruct" does not exist on Hugging Face
    embedding_model: str = "BAAI/bge-small-en-v1.5"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    top_k: int = Field(5, ge=1, le=20, description="Evidence passages given to Qwen")
    max_context_chars: int = Field(7000, ge=500)
    eval_limit: int = Field(0, ge=0, description="0 runs every gold question")

CFG = Settings(**{field: value for field, value in {"work_dir": os.getenv("RAG_WORK_DIR"), "qwen_model": os.getenv("QWEN_MODEL"),
                                                    "eval_limit": os.getenv("EVAL_LIMIT")}.items() if value})
CFG.work_dir.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0
print(f"Device: {DEVICE}, VRAM: {VRAM_GB:.1f} GB")
if DEVICE == "cpu":
    print("WARNING: no GPU. In Colab choose Runtime > Change runtime type > T4 GPU; Qwen will not load in 4-bit without it.")

def locate_pdf():
    if os.getenv("PDF_PATH"):
        return Path(os.environ["PDF_PATH"])
    found = sorted(glob.glob(str(CFG.work_dir / "*.pdf")))
    expected = [p for p in found if Path(p).name == EXPECTED_PDF]
    if expected or found:
        return Path((expected or found)[0])
    from google.colab import files
    print(f"Upload the annual report PDF ({EXPECTED_PDF} for the gold evaluation).")
    uploaded = files.upload()
    name = next(n for n in uploaded if n.lower().endswith(".pdf"))
    return Path(shutil.move(name, CFG.work_dir / name)) if Path(name).resolve().parent != CFG.work_dir.resolve() else CFG.work_dir / name

PDF_PATH = locate_pdf()
PDF_SHA = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
GOLD_VALID = PDF_SHA == EXPECTED_SHA256
print(f"PDF: {PDF_PATH} ({PDF_PATH.stat().st_size / 1e6:.1f} MB)")
print("Gold evaluation applies to this PDF." if GOLD_VALID else
      "This is not the PDF the gold questions were written for: you can ask your own questions, but the evaluation cell will be skipped.")

## Step 3: Parse text, tables and scanned pages
Each printed line is rebuilt from word positions, so table rows keep their columns, and each table chunk repeats its year header and unit. Pages with no usable text layer, including ones whose fonts extract as garbage, go to OCR.

In [ ]:
import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

NUMERIC_CELL = re.compile(r"^(\$\s*)?\(?-?\d[\d,]*(\.\d+)?\)?\s*%?$|^[—–-]$")
YEAR_CELL = re.compile(r"^(19|20)\d{2}$")
UNIT_LINE = re.compile(r"(millions? of|billions? of|thousands? of)[^|]{0,50}", re.I)
_ocr_engine = None

class TableSegment(BaseModel):
    header: str = ""
    unit: str = ""
    rows: list[str] = Field(min_length=1)

class ParsedPage(BaseModel):
    page: int = Field(ge=1)
    narrative: str = ""
    tables: list[TableSegment] = Field(default_factory=list)
    ocr_used: bool = False
    visual: bool = False

class ChunkMetadata(BaseModel):
    page: int = Field(ge=1)
    kind: Literal["table", "text", "page"]
    ocr_used: bool = False
    chunk_id: int = -1

def chunk(text, page, kind, ocr_used=False):
    return Document(page_content=text, metadata=ChunkMetadata(page=page, kind=kind, ocr_used=ocr_used).model_dump())

def looks_garbled(text):
    """A broken font map extracts as control characters and symbols, which is not an empty page."""
    if not text.strip():
        return False
    readable = sum(c.isalnum() or c in " \n\t.,$%()-:;'\"/&’–—‑" for c in text) / len(text)
    return readable < 0.85 or len(re.findall(r"[A-Za-z]{3,}", text)) < len(text) / 40

def ocr_words(page):
    """RapidOCR lines as (x0, y0, x1, y1, text), the same shape as PyMuPDF words."""
    global _ocr_engine
    import numpy as np
    from rapidocr import RapidOCR
    _ocr_engine = _ocr_engine or RapidOCR(params={"Global.log_level": "error"})
    pix = page.get_pixmap(dpi=200, colorspace=pymupdf.csRGB, alpha=False)
    result = _ocr_engine(np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3))
    if result.boxes is None:
        return []
    ratio = 72 / 200
    return [(min(p[0] for p in box) * ratio, min(p[1] for p in box) * ratio, max(p[0] for p in box) * ratio,
             max(p[1] for p in box) * ratio, text) for box, text, score in zip(result.boxes, result.txts, result.scores) if score >= 0.6]

def layout_rows(words):
    """Rebuild printed lines from word positions; a wide gap starts a new cell, so table columns survive."""
    words = sorted(words, key=lambda w: (round(w[3]), w[0]))
    lines, seen = [], set()
    for word in words:
        key = (word[4], round(word[0] / 3), round(word[3] / 3))  # bold is often printed twice at the same spot
        if key in seen:
            continue
        seen.add(key)
        if lines and abs(lines[-1][0] - word[3]) < 3.5:
            lines[-1][1].append(word)
        else:
            lines.append([word[3], [word]])
    rows = []
    for _, line in lines:
        line.sort(key=lambda w: w[0])
        cells, current = [], [line[0]]
        for word in line[1:]:
            if word[0] - current[-1][2] > 9:
                cells.append(current)
                current = [word]
            else:
                current.append(word)
        cells.append(current)
        joined = [" ".join(w[4] for w in cell).strip() for cell in cells]
        rows.append([c for c in joined if c and c != "$"])
    return rows

def is_table_row(cells):
    return sum(bool(NUMERIC_CELL.match(c)) for c in cells) >= 2

def table_segments(rows):
    """Group table rows; keep the year header and unit with each table so a row can never lose its columns."""
    segments, current, header, unit, gap, previous = [], [], "", "", 0, None
    for cells in rows:
        text = " | ".join(cells)
        if current and " | " in current[-1] and re.fullmatch(r"\([A-Za-z ,-]+\)", text):
            label, values = current[-1].split(" | ", 1)
            current[-1] = f"{label} {text} | {values}"
            continue
        match = UNIT_LINE.search(text)
        if match and not is_table_row(cells):
            unit = match.group(0).strip(" )")
        if sum(bool(YEAR_CELL.match(c)) for c in cells) >= 2:
            header = " | ".join(cells)
            previous = None
            continue
        if is_table_row(cells):
            # A two-line label sits on a different baseline from its values: join them so the row keeps its name.
            if not any(re.search(r"[A-Za-z]{3}", c) for c in cells):
                if current and not re.search(r"\d", current[-1]) and " | " not in current[-1]:
                    text = f"{current.pop()} | {text}"
                elif not current and previous and not is_table_row(previous) and len(" ".join(previous)) < 75:
                    text = f"{' '.join(previous)} | {text}"
            current.append(text)
            gap = 0
        elif current and len(text) < 75 and gap < 2:
            current.append(text)  # sub-labels and group headings inside a table
            gap += 1
        elif current:
            segments.append(TableSegment(header=header, unit=unit, rows=current))
            current, gap = [], 0
        previous = cells
    if current:
        segments.append(TableSegment(header=header, unit=unit, rows=current))
    return [s for s in segments if sum(" | " in r and bool(re.search(r"[A-Za-z]{3}", r)) for r in s.rows) >= 2]

def has_visual(page, text):
    area = page.rect.width * page.rect.height
    picture = any(0.05 < r.width * r.height / area < 0.85 for image in page.get_images(full=True) for r in page.get_image_rects(image[0]))
    fills = [d for d in page.get_drawings() if d.get("fill") and d["rect"].width * d["rect"].height > 30]
    return picture or (15 <= len(fills) <= 900 and len(re.findall(r"\d", text)) > 10 and len(text) < 3500)

def parse_pdf(path):
    pages = []
    with pymupdf.open(path) as doc:
        for number, page in enumerate(doc, start=1):
            raw = page.get_text()
            ocr_used = len(raw.strip()) < 100 or looks_garbled(raw)
            words = ocr_words(page) if ocr_used else [w[:5] for w in page.get_text("words")]
            rows = layout_rows(words) if words else []
            segments = table_segments(rows)
            in_table = {cell for s in segments for row in s.rows for cell in row.split(" | ")}
            if ocr_used:
                narrative = "\n".join(" ".join(r) for r in rows if not is_table_row(r))
            else:
                blocks = [b[4] for b in sorted(page.get_text("blocks"), key=lambda b: (b[1], b[0])) if b[6] == 0]
                narrative = "\n".join(x.strip() for b in blocks for x in b.splitlines()
                                      if x.strip() and not (segments and (x.strip() in in_table or NUMERIC_CELL.match(x.strip()) or x.strip() == "$")))
            narrative = re.sub(r"[ \t]+", " ", narrative).strip()
            pages.append(ParsedPage(page=number, narrative=narrative, tables=segments, ocr_used=ocr_used, visual=has_visual(page, raw)))
    return pages

def linear_rows(segment):
    """Write each value beside its year and each row under its group, so nothing depends on counting columns."""
    cells = segment.header.split(" | ")
    first_year = next((i for i, cell in enumerate(cells) if YEAR_CELL.match(cell)), None)
    years = cells[first_year:] if first_year is not None else []
    if len(years) < 2 or len(set(years)) != len(years) or not all(YEAR_CELL.match(y) for y in years):
        return segment.rows  # no year columns, or repeated ones such as one set per segment: leave as printed
    rows, group = [], ""
    for row in segment.rows:
        parts = row.split(" | ")
        split_label = re.fullmatch(r"(\$?\s*\(?[\d,.]+\)?)\s+(\(.+\))", parts[0])
        if split_label:  # a value that landed on the same baseline as its unit, under a two-line label
            parts = [f"{group} {split_label.group(2)}".strip(), split_label.group(1), *parts[1:]]
            group = ""
        if len(parts) == 1:
            group = row if not re.search(r"\d\s*$", row) or re.search(r"[A-Za-z]\d$", row) else group
            rows.append(row)
            continue
        values = parts[-len(years):]
        label = " ".join(parts[:-len(years)]).strip()
        if len(parts) <= len(years) or not all(NUMERIC_CELL.match(v) for v in values):
            rows.append(row)
            continue
        named = f"{group} > {label}" if group and not label.lower().startswith(("total", group.lower())) and not label.startswith("(") else label
        rows.append(f"{named}: " + "; ".join(f"{y}={v}" for y, v in zip(years, values)))
        if label.lower().startswith("total"):
            group = ""
    return rows

def table_chunks(segment, max_chars=1400):
    """Split long tables between rows, repeating the column header and unit in every piece."""
    head = "\n".join(x for x in [f"Units: {segment.unit}" if segment.unit else "",
                                 f"Columns: {segment.header}" if segment.header else ""] if x)
    pieces, current = [], []
    for row in linear_rows(segment):
        if current and sum(map(len, current)) + len(head) + len(row) > max_chars:
            pieces.append(head + "\n" + "\n".join(current))
            current = []
        current.append(row)
    if current:
        pieces.append(head + "\n" + "\n".join(current))
    return [p.strip() for p in pieces]

def make_chunks(pages):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, separators=["\n\n", "\n", ". ", " "])
    chunks = []
    for item in pages:
        for segment in item.tables:
            chunks += [chunk(text, item.page, "table", item.ocr_used) for text in table_chunks(segment)]
        if len(item.narrative) >= 40:
            chunks += [chunk(text, item.page, "text", item.ocr_used) for text in splitter.split_text(item.narrative)]
    for index, doc in enumerate(chunks):
        doc.metadata["chunk_id"] = index
    return chunks

def page_text(item):
    tables = "\n\n".join("\n".join(table_chunks(s, 10**6)) for s in item.tables)
    return "\n\n".join(x for x in [tables, item.narrative] if x)

started = time.time()
PAGES = parse_pdf(PDF_PATH)
CHUNKS = make_chunks(PAGES)
print(f"Parsed {len(PAGES)} pages into {len(CHUNKS)} chunks in {time.time() - started:.0f}s: "
      f"{sum(p.ocr_used for p in PAGES)} pages needed OCR, {sum(bool(p.tables) for p in PAGES)} have tables, "
      f"{sum(p.visual for p in PAGES)} have pictures or charts.")
print(next(c.page_content for c in CHUNKS if c.metadata["kind"] == "table")[:400])

## Step 4: Hybrid retriever with reranking
BM25 and Chroma candidates merged with RRF (k=60), then reranked. If even the best passage scores under the floor of -4, the question counts as out of scope.

In [ ]:
from langchain_community.retrievers import BM25Retriever

STOPWORDS = set("a an and are at by did do does for from had has have how in is it its of on or per the to was were what when which who with".split())

def bm25_tokens(text):
    """Keep figures whole ("17,661", "10.4") so exact-number and exact-term search works."""
    return [t for t in re.findall(r"[a-z0-9]+(?:[.,'&-][a-z0-9]+)*", text.lower()) if t not in STOPWORDS]

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDINGS = HuggingFaceEmbeddings(model_name=CFG.embedding_model, model_kwargs={"device": DEVICE},
                                   encode_kwargs={"normalize_embeddings": True})

def build_chroma(chunks, name):
    """A fresh in-memory collection, so rerunning a cell can never mix old and new chunks."""
    store = Chroma(collection_name=f"{name}_{time.time_ns()}", embedding_function=EMBEDDINGS, collection_metadata={"hnsw:space": "cosine"})
    store.add_documents(chunks, ids=[str(c.metadata["chunk_id"]) for c in chunks])
    return store

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

RERANK_FLOOR = -4.0  # raw logit; the best hit must clear it or the question is treated as out of scope
CROSS_ENCODER = HuggingFaceCrossEncoder(model_name=CFG.reranker_model, model_kwargs={"device": "cpu"})

def rrf_merge(result_lists, k=60):
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
            docs[key] = doc
    return [docs[key] for key in sorted(scores, key=scores.get, reverse=True)]

def rerank(query, docs, top_k=CFG.top_k):
    """Score (question, passage) pairs. Judge only the best score, so a low-scoring table beside a good hit stays."""
    if not docs:
        return []
    scores = [float(s) for s in CROSS_ENCODER.score([(query, d.page_content) for d in docs])]
    if max(scores) < RERANK_FLOOR:
        return []
    ranked = sorted(zip(scores, docs), key=lambda pair: pair[0], reverse=True)[:top_k]
    return [Document(page_content=d.page_content, metadata={**d.metadata, "rerank_score": s}) for s, d in ranked]

In [ ]:
from typing import Any, List
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.retrievers import BaseRetriever

class HybridRerankRetriever(BaseRetriever):
    bm25: Any
    dense: Any
    candidates: int = 30

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        return rerank(query, rrf_merge([self.bm25.invoke(query), self.dense.invoke(query)]))

RETRIEVER = HybridRerankRetriever(
    bm25=BM25Retriever.from_documents(CHUNKS, k=30, preprocess_func=bm25_tokens),
    dense=build_chroma(CHUNKS, "hybrid").as_retriever(search_kwargs={"k": 30}))
print("Hybrid index ready:", len(CHUNKS), "chunks")

## Step 5: Load Qwen in 4-bit through LangChain
`Qwen2.5-7B-Instruct` takes about 5.6 GB of VRAM at 4-bit. Set the `QWEN_MODEL` environment variable to try another model, such as `Qwen/Qwen3-8B`, but note that `Qwen/Qwen2.5-8B-Instruct` isn't a published model.

In [ ]:
from functools import partial

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(CFG.qwen_model)
if "qwen3" in CFG.qwen_model.lower():  # thinking would break the JSON reply and use up the token limit
    tokenizer.apply_chat_template = partial(tokenizer.apply_chat_template, enable_thinking=False)
model = AutoModelForCausalLM.from_pretrained(CFG.qwen_model, quantization_config=quant, device_map={"": 0})  # 4-bit weights must stay on the GPU
# No repetition_penalty: it pushes the model away from digits already in the context, which corrupts copied figures.
text_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=140, do_sample=False,
                     return_full_text=False, pad_token_id=tokenizer.eos_token_id)
for attribute in ("temperature", "top_p", "top_k"):
    setattr(model.generation_config, attribute, None)
CHAT_LLM = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_pipe), model_id=CFG.qwen_model, tokenizer=tokenizer)
print(f"Loaded {CFG.qwen_model}; GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Step 6: Grounded answer chain and fallback
The LCEL chain is `prompt | Qwen | parser`, and `PydanticOutputParser` validates the reply against `LLMAnswer`. The gates then withhold any answer where:
- the reply doesn't validate against the schema,
- a figure doesn't appear in the evidence,
- a citation points outside the evidence, or
- the question's year isn't in the evidence.

When the LLM refuses or an answer is withheld, `final_answer` becomes the gold answer for a gold question. For any other question it becomes the closest passage, clearly labelled as not a confirmed answer.

In [ ]:
class GoldBase(BaseModel):
    model_config = ConfigDict(frozen=True, extra="forbid")
    id: str
    question: str = Field(min_length=5)
    ground_truth: str = Field(min_length=1)
    pages: list[int] = Field(default_factory=list)

class NumericGold(GoldBase):
    type: Literal["numeric"]
    kind: Literal["table", "narrative", "chart"]
    value: str
    scale: int = Field(1, gt=0)

    @field_validator("value")
    @classmethod
    def value_is_a_number(cls, value):
        Decimal(value.replace(",", ""))
        return value

    @property
    def decimal(self):
        return Decimal(self.value.replace(",", ""))

class TextGold(GoldBase):
    type: Literal["text"]
    kind: Literal["text"]
    keywords: list[str] = Field(min_length=1)

class NegativeGold(GoldBase):
    type: Literal["negative"]
    kind: Literal["negative"]

GoldQuestion = Annotated[Union[NumericGold, TextGold, NegativeGold], Field(discriminator="type")]

GOLD_SET = TypeAdapter(list[GoldQuestion]).validate_python([
{
"id": "T01",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's net income attributable to Chevron Corporation in 2024?",
"value": "17,661",
"scale": 1000000,
"pages": [
22,
27
],
"ground_truth": "17,661 million US dollars [p.22]"
},
{
"id": "T02",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's sales and other operating revenues in 2023?",
"value": "196,913",
"scale": 1000000,
"pages": [
22,
27,
56
],
"ground_truth": "196,913 million US dollars [p.22]"
},
{
"id": "T03",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's cash flow from operating activities in 2022?",
"value": "49,602",
"scale": 1000000,
"pages": [
22,
59
],
"ground_truth": "49,602 million US dollars [p.22]"
},
{
"id": "T04",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's capital expenditures in 2024?",
"value": "16,448",
"scale": 1000000,
"pages": [
22,
59
],
"ground_truth": "16,448 million US dollars [p.22]"
},
{
"id": "T05",
"type": "numeric",
"kind": "table",
"question": "What were Chevron's total assets at year-end 2023?",
"value": "261,632",
"scale": 1000000,
"pages": [
22,
58
],
"ground_truth": "261,632 million US dollars [p.22]"
},
{
"id": "T06",
"type": "numeric",
"kind": "table",
"question": "What was Chevron Corporation stockholders' equity at year-end 2024?",
"value": "152,318",
"scale": 1000000,
"pages": [
22,
58
],
"ground_truth": "152,318 million US dollars [p.22]"
},
{
"id": "T07",
"type": "numeric",
"kind": "table",
"question": "What were diluted earnings per share in 2024?",
"value": "9.72",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "9.72 US dollars per share [p.22]"
},
{
"id": "T08",
"type": "numeric",
"kind": "table",
"question": "What were cash dividends per share in 2023?",
"value": "6.04",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "6.04 US dollars per share [p.22]"
},
{
"id": "T09",
"type": "numeric",
"kind": "table",
"question": "What was Chevron's net debt ratio in 2024?",
"value": "10.4",
"scale": 1,
"pages": [
22
],
"ground_truth": "10.4 percent [p.22]"
},
{
"id": "T10",
"type": "numeric",
"kind": "table",
"question": "What was the return on average capital employed in 2022?",
"value": "20.3",
"scale": 1,
"pages": [
22,
27
],
"ground_truth": "20.3 percent [p.22]"
},
{
"id": "T11",
"type": "numeric",
"kind": "table",
"question": "What was net production of crude oil, condensate and synthetic oil in 2024, in thousands of barrels per day?",
"value": "1,560",
"scale": 1,
"pages": [
23
],
"ground_truth": "1,560 thousand barrels per day [p.23]"
},
{
"id": "T12",
"type": "numeric",
"kind": "table",
"question": "What was net production of natural gas liquids in 2022, in thousands of barrels per day?",
"value": "279",
"scale": 1,
"pages": [
23
],
"ground_truth": "279 thousand barrels per day [p.23]"
},
{
"id": "T13",
"type": "numeric",
"kind": "table",
"question": "What was net production of natural gas in 2023, in millions of cubic feet per day?",
"value": "7,744",
"scale": 1,
"pages": [
23
],
"ground_truth": "7,744 million cubic feet per day [p.23]"
},
{
"id": "T14",
"type": "numeric",
"kind": "table",
"question": "What were net proved oil-equivalent reserves at year-end 2024, in millions of barrels?",
"value": "9,804",
"scale": 1,
"pages": [
23
],
"ground_truth": "9,804 million barrels [p.23]"
},
{
"id": "T15",
"type": "numeric",
"kind": "table",
"question": "How many employees did Chevron have at year-end 2024, excluding service station employees?",
"value": "39,742",
"scale": 1,
"pages": [
23
],
"ground_truth": "39,742 employees [p.23]"
},
{
"id": "T16",
"type": "numeric",
"kind": "table",
"question": "How many service station employees were excluded from the 2024 employee count?",
"value": "5,556",
"scale": 1,
"pages": [
23
],
"ground_truth": "5,556 service station employees [p.23]"
},
{
"id": "T17",
"type": "numeric",
"kind": "table",
"question": "What were total upstream earnings in 2024?",
"value": "18,602",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "18,602 million US dollars [p.27]"
},
{
"id": "T18",
"type": "numeric",
"kind": "table",
"question": "What were United States downstream earnings in 2023?",
"value": "3,904",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "3,904 million US dollars [p.27]"
},
{
"id": "T19",
"type": "numeric",
"kind": "table",
"question": "What was the All Other earnings line in 2022?",
"value": "2,974",
"scale": 1000000,
"pages": [
27
],
"ground_truth": "2,974 million US dollars (a loss) [p.27]"
},
{
"id": "T20",
"type": "numeric",
"kind": "table",
"question": "What was goodwill on the balance sheet at December 31, 2024?",
"value": "4,578",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "4,578 million US dollars [p.58]"
},
{
"id": "T21",
"type": "numeric",
"kind": "table",
"question": "What were total liabilities at December 31, 2024?",
"value": "103,781",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "103,781 million US dollars [p.58]"
},
{
"id": "T22",
"type": "numeric",
"kind": "table",
"question": "What was long-term debt at December 31, 2023?",
"value": "20,307",
"scale": 1000000,
"pages": [
58
],
"ground_truth": "20,307 million US dollars [p.58]"
},
{
"id": "T23",
"type": "numeric",
"kind": "table",
"question": "What was depreciation, depletion and amortization in the 2024 cash flow statement?",
"value": "17,282",
"scale": 1000000,
"pages": [
59
],
"ground_truth": "17,282 million US dollars [p.59]"
},
{
"id": "T24",
"type": "numeric",
"kind": "table",
"question": "What was net cash used for investing activities in 2023?",
"value": "15,232",
"scale": 1000000,
"pages": [
59
],
"ground_truth": "15,232 million US dollars [p.59]"
},
{
"id": "T25",
"type": "numeric",
"kind": "table",
"question": "What were total revenues and other income in 2024?",
"value": "202,792",
"scale": 1000000,
"pages": [
56
],
"ground_truth": "202,792 million US dollars [p.56]"
},
{
"id": "T26",
"type": "numeric",
"kind": "table",
"question": "What was the effective income tax rate in 2024?",
"value": "35.5",
"scale": 1,
"pages": [
79
],
"ground_truth": "35.5 percent [p.79]"
},
{
"id": "N01",
"type": "numeric",
"kind": "narrative",
"question": "What did Permian Basin production average in 2024, in net barrels of oil-equivalent per day?",
"value": "921,000",
"scale": 1,
"pages": [
7
],
"ground_truth": "921,000 net boe per day [p.7]"
},
{
"id": "N02",
"type": "numeric",
"kind": "narrative",
"question": "Up to what pressure can the Anchor project technology operate, in pounds per square inch?",
"value": "20,000",
"scale": 1,
"pages": [
2,
7
],
"ground_truth": "20,000 pounds per square inch [p.2]"
},
{
"id": "N03",
"type": "numeric",
"kind": "narrative",
"question": "How many gross wells were added to Chevron's Gulf of America production in 2024?",
"value": "17",
"scale": 1,
"pages": [
2
],
"ground_truth": "17 gross wells [p.2]"
},
{
"id": "N04",
"type": "numeric",
"kind": "narrative",
"question": "What is the design capacity of the Anchor FPU in gross barrels of oil per day?",
"value": "75,000",
"scale": 1,
"pages": [
3
],
"ground_truth": "75,000 gross barrels of oil per day [p.3]"
},
{
"id": "N05",
"type": "numeric",
"kind": "narrative",
"question": "How many thousand net barrels of oil-equivalent per day came from Chevron's Gulf of America assets in 2024?",
"value": "192",
"scale": 1,
"pages": [
2
],
"ground_truth": "192 thousand net boe per day [p.2]"
},
{
"id": "N06",
"type": "numeric",
"kind": "narrative",
"question": "What total assets figure, in billions of dollars, does the Chevron at a glance page show?",
"value": "256.9",
"scale": 1,
"pages": [
18
],
"ground_truth": "256.9 billion US dollars [p.18]"
},
{
"id": "C01",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in Chevron stock worth at the end of 2024?",
"value": "150",
"scale": 1,
"pages": [
21
],
"ground_truth": "150 US dollars [p.21]"
},
{
"id": "C02",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in the S&P 500 worth at the end of 2024?",
"value": "197",
"scale": 1,
"pages": [
21
],
"ground_truth": "197 US dollars [p.21]"
},
{
"id": "C03",
"type": "numeric",
"kind": "chart",
"question": "In the five-year cumulative total returns graph, what was $100 invested in the peer group worth at the end of 2024?",
"value": "153",
"scale": 1,
"pages": [
21
],
"ground_truth": "153 US dollars [p.21]"
},
{
"id": "C04",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 1-year annualized return as of 12/31/2024?",
"value": "1.4",
"scale": 1,
"pages": [
20
],
"ground_truth": "1.4 percent [p.20]"
},
{
"id": "C05",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 5-year annualized return as of 12/31/2024?",
"value": "8.5",
"scale": 1,
"pages": [
20
],
"ground_truth": "8.5 percent [p.20]"
},
{
"id": "C06",
"type": "numeric",
"kind": "chart",
"question": "According to the total stockholder returns bar chart, what was Chevron's 10-year annualized return as of 12/31/2024?",
"value": "7.0",
"scale": 1,
"pages": [
20
],
"ground_truth": "7.0 percent [p.20]"
},
{
"id": "C07",
"type": "numeric",
"kind": "chart",
"question": "What S&P 500 10-year return does the total stockholder returns chart mark?",
"value": "13.1",
"scale": 1,
"pages": [
20
],
"ground_truth": "13.1 percent [p.20]"
},
{
"id": "C08",
"type": "numeric",
"kind": "chart",
"question": "What compound annual growth rate does the indexed dividend growth chart show for Chevron?",
"value": "6.2",
"scale": 1,
"pages": [
20
],
"ground_truth": "6.2 percent [p.20]"
},
{
"id": "F01",
"type": "text",
"kind": "text",
"question": "Who is Chevron's Chairman of the Board and Chief Executive Officer?",
"keywords": [
"wirth"
],
"pages": [
12
],
"ground_truth": "Michael K. (Mike) Wirth [p.12]"
},
{
"id": "F02",
"type": "text",
"kind": "text",
"question": "Who is Chevron's Lead Director?",
"keywords": [
"austin"
],
"pages": [
12
],
"ground_truth": "Wanda M. Austin [p.12]"
},
{
"id": "F03",
"type": "text",
"kind": "text",
"question": "Which independent registered public accounting firm audited Chevron's financial statements?",
"keywords": [
"pricewaterhouse"
],
"pages": [
53,
55
],
"ground_truth": "PricewaterhouseCoopers LLP [p.53]"
},
{
"id": "F04",
"type": "text",
"kind": "text",
"question": "What rating does Standard and Poor's give the bonds issued or guaranteed by Chevron Corporation?",
"keywords": [
"aa-"
],
"pages": [
41
],
"ground_truth": "AA- [p.41]"
},
{
"id": "F05",
"type": "text",
"kind": "text",
"question": "Which companies are in the peer group used in the stock performance charts?",
"keywords": [
"bp",
"exxon",
"shell",
"total"
],
"pages": [
20,
21
],
"ground_truth": "BP, ExxonMobil, Shell and TotalEnergies [p.20]"
},
{
"id": "F06",
"type": "text",
"kind": "text",
"question": "Which company did Chevron agree to acquire in an all-stock merger announced in October 2023?",
"keywords": [
"hess"
],
"pages": [
95
],
"ground_truth": "Hess Corporation [p.95]"
},
{
"id": "X01",
"type": "negative",
"kind": "negative",
"question": "How much bitcoin does Chevron hold on its balance sheet?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X02",
"type": "negative",
"kind": "negative",
"question": "What was Tesla's total revenue in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X03",
"type": "negative",
"kind": "negative",
"question": "What was ExxonMobil's net income in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X04",
"type": "negative",
"kind": "negative",
"question": "What was Apple Inc.'s revenue in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X05",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's net income in 2021?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X06",
"type": "negative",
"kind": "negative",
"question": "What were Chevron's sales and other operating revenues in 2020?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X07",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's net income in 2025?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X08",
"type": "negative",
"kind": "negative",
"question": "What capital expenditures did Chevron report for 2018?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X09",
"type": "negative",
"kind": "negative",
"question": "What was Michael Wirth's total compensation in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X10",
"type": "negative",
"kind": "negative",
"question": "How many electric vehicle charging stations does Chevron operate?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X11",
"type": "negative",
"kind": "negative",
"question": "What percentage of Chevron's employees are women?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X12",
"type": "negative",
"kind": "negative",
"question": "How many solar panels did Chevron install in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X13",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's closing share price on December 31, 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X14",
"type": "negative",
"kind": "negative",
"question": "What was the gross margin on Chevron's lithium business in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X15",
"type": "negative",
"kind": "negative",
"question": "What is the home address of Chevron's Chief Financial Officer?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X16",
"type": "negative",
"kind": "negative",
"question": "What is Chevron's forecast net income for 2027?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X17",
"type": "negative",
"kind": "negative",
"question": "How many offshore wind turbines did Chevron build in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X18",
"type": "negative",
"kind": "negative",
"question": "How many retail coffee shops does Chevron own in Japan?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X19",
"type": "negative",
"kind": "negative",
"question": "What was Chevron's employee attrition rate in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
},
{
"id": "X20",
"type": "negative",
"kind": "negative",
"question": "What was Shell's total production in 2024?",
"pages": [],
"ground_truth": "This report does not contain this information."
}
])
GOLD_BY_QUESTION = {row.question.strip().lower(): row for row in GOLD_SET}

from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser

class LLMAnswer(BaseModel):
    """The only output accepted from Qwen. Anything else fails closed."""
    in_report: bool = Field(description="True only if the evidence states the item for the asked company and year")
    answer: Optional[str] = Field(None, description="One short sentence with the figure as printed")
    cited_pages: list[int] = Field(default_factory=list, description="PDF pages from the evidence labels")

    @model_validator(mode="after")
    def answer_matches_flag(self):
        if self.in_report and not (self.answer or "").strip():
            raise ValueError("in_report is true but there is no answer")
        if not self.in_report:
            self.answer, self.cited_pages = None, []
        return self

class GateDecision(BaseModel):
    answer: Optional[str] = None
    reason: str = "ok"
    cited_pages: list[int] = Field(default_factory=list)

Status = Literal["LLM_ANSWER", "GROUND_TRUTH_FALLBACK"]
Source = Literal["llm", "gold_set", "closest_passage", "none"]

class GroundedResult(BaseModel):
    question: str
    status: Status
    final_answer: str
    final_source: Source
    llm_answer: Optional[str] = None
    llm_raw: str = ""
    withheld_reason: str = ""
    cited_pages: list[int] = Field(default_factory=list)
    retrieved_pages: list[int] = Field(default_factory=list)
    context: str = ""
    retrieval_s: float = Field(ge=0)
    total_s: float = Field(ge=0)

    @model_validator(mode="after")
    def fallback_is_never_llm(self):
        if (self.status == "LLM_ANSWER") != (self.final_source == "llm") or (self.status == "LLM_ANSWER") != bool(self.llm_answer):
            raise ValueError("status, source and llm_answer disagree")
        return self

SYSTEM_PROMPT = """You answer questions about one company's annual report using ONLY the evidence passages.

Reply with one JSON object and nothing else, with these keys:
"in_report": true or false
"answer": one short sentence, or null
"cited_pages": the page numbers from the evidence labels, such as [12]

Rules:
1. Copy figures exactly as printed, with the unit shown in the evidence. Do not calculate, round or estimate.
2. In a table, take the value from the row that matches the item, and within that row the value after the year asked about, such as 2023=388. "Group > Item" means the item belongs to that group.
3. The evidence must state the item for the same company and the same year as the question. If it does not, set "in_report" to false and "answer" to null. Never substitute another year, company or similar item.

Examples:
Evidence: [p.4] Units: Millions of dollars
Total widgets revenue: 2024=410; 2023=388
Question: What was widgets revenue in 2023?
Answer: {{"in_report": true, "answer": "Widgets revenue was $388 million in 2023.", "cited_pages": [4]}}

Evidence: [p.4] Units: Millions of dollars
Total widgets revenue: 2024=410; 2023=388
Question: What was widgets revenue in 2019?
Answer: {{"in_report": false, "answer": null, "cited_pages": []}}

Evidence: [p.9] The board includes the chief executive of Globex Inc.
Question: What was Globex Inc.'s revenue?
Answer: {{"in_report": false, "answer": null, "cited_pages": []}}"""

ANSWER_CHAIN = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT), ("human", "Evidence:\n{context}\n\nQuestion: {question}\nAnswer:"),
]) | CHAT_LLM | StrOutputParser()
ANSWER_PARSER = PydanticOutputParser(pydantic_object=LLMAnswer)

CITATION = re.compile(r"\[?\(?\b(?:p\.|pp\.|page)\s*(\d+)\)?\]?", re.I)
NUMBER = re.compile(r"(?<![\w.,])\(?\$?\(?(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\)?\s*(%|thousand|million|billion|trillion)?", re.I)
SCALES = {"thousand": Decimal(10) ** 3, "million": Decimal(10) ** 6, "billion": Decimal(10) ** 9, "trillion": Decimal(10) ** 12}
REFUSAL_WORDS = re.compile(r"\b(does|do|did) not (contain|provide|include|mention|state|specify|disclose|show)\b|"
                           r"\bnot (provided|available|mentioned|disclosed|stated|specified|found|included)\b|\bno (information|data|mention)\b|"
                           r"\bcannot (be )?(determine|determined|answer|find)\b|\bunable to\b", re.I)

def numbers_in(text):
    """(digits as printed, value without scale, scale). Bare years and page citations are not figures."""
    found = []
    for match in NUMBER.finditer(CITATION.sub(" ", text)):
        printed = match.group(1)
        value = Decimal(printed.replace(",", ""))
        if "," not in printed and "." not in printed and 1900 <= value <= 2100 and not match.group(2):
            continue
        found.append((printed, value, SCALES.get((match.group(2) or "").lower(), Decimal(1))))
    return found

def same_figure(printed, value, scale, gold, gold_scale=Decimal(1)):
    """Exact, or a correct rounding to the printed decimals with at least 3 significant digits: $17.7 billion matches 17,661 million."""
    if value == gold:
        return True
    target = gold * gold_scale / scale
    decimals = len(printed.split(".")[1]) if "." in printed else 0
    if value == target:
        return True
    significant = len(printed.replace(",", "").replace(".", "").lstrip("0"))
    return significant >= 3 and scale != Decimal(1) and target.quantize(Decimal(1).scaleb(-decimals), rounding=ROUND_HALF_UP) == value

def figure_in_evidence(printed, value, scale, evidence_numbers):
    for e_printed, e_value, e_scale in evidence_numbers:
        if same_figure(printed, value, Decimal(1), e_value) or same_figure(printed, value, scale, e_value, e_scale):
            return True
        for power in (3, 6, 9):  # evidence in millions, answer in billions and the like
            if same_figure(printed, value, scale, e_value, Decimal(10) ** power):
                return True
    return False

QUESTION_STOPWORDS = {"what", "was", "were", "which", "who", "how", "many", "much", "did", "does", "the", "and", "for", "from", "with", "that",
                      "this", "has", "had", "its", "are", "chevron", "corporation", "company", "report", "year", "end", "per", "day", "total"}

def content_words(text):
    return {word.rstrip("s") for word in re.findall(r"[a-z]{3,}", text.lower()) if word not in QUESTION_STOPWORDS}

def table_cells(line):
    return [cell.strip() for cell in line.replace("Columns:", "").split("|") if cell.strip()]

def year_of(cell):
    years = re.findall(r"\b(?:19|20)\d{2}\b", cell)
    return years[-1] if len(years) == 1 and len(cell) < 30 else None

def in_year_column(header, cells, figure, year):
    """Values align with the header from the right. If the counts differ the row can't be checked, so it isn't blocked."""
    printed, value, scale = figure
    columns = header[next(i for i, cell in enumerate(header) if year_of(cell)):]
    values = [cell for cell in cells if re.fullmatch(r"[$(\s]*[\d,.]+[)%\s]*", cell)]
    if len(values) != len(columns):
        return True
    return any(year_of(column) == year and figure_in_evidence(printed, value, scale, numbers_in(cell))
               for column, cell in zip(columns, values))

def figure_placement(question, figure, docs):
    """Printed somewhere is not enough: the figure must sit beside the asked item and, in a table, under the asked year."""
    printed, value, scale = figure
    words = content_words(question)
    years = re.findall(r"\b(?:19|20)\d{2}\b", question)
    year = years[0] if len(set(years)) == 1 else None
    outcome = "figure_not_beside_item"
    for doc in docs:
        lines, header = doc.page_content.splitlines(), None
        for index, line in enumerate(lines):
            cells = table_cells(line)
            if sum(bool(year_of(cell)) for cell in cells) >= 2:
                header = cells
                continue
            if not figure_in_evidence(printed, value, scale, numbers_in(line)):
                continue
            nearby = " ".join(lines[max(0, index - 2):index + 2])
            if words and not content_words(nearby) & words:
                continue
            pairs = re.findall(r"\b((?:19|20)\d{2})=([^;]+)", line)
            if year and pairs and not any(y == year and figure_in_evidence(printed, value, scale, numbers_in(v)) for y, v in pairs):
                outcome = "figure_in_other_year_column"
                continue
            if year and header and len(cells) > 1 and not pairs and not in_year_column(header, cells, figure, year):
                outcome = "figure_in_other_year_column"
                continue
            return "ok"
    return outcome

def evidence_block(docs):
    parts, used = [], 0
    for doc in docs:
        part = f"[p.{doc.metadata['page']}] {doc.page_content}"
        if used + len(part) > CFG.max_context_chars:
            part = part[:max(0, CFG.max_context_chars - used)]
        if part:
            parts.append(part)
            used += len(part)
    return "\n\n---\n\n".join(parts)

def gate_answer(question, raw, docs):
    """Parse against the schema, then run deterministic checks on the result."""
    try:
        parsed = ANSWER_PARSER.parse(raw or "")
    except OutputParserException:
        return GateDecision(reason="invalid_llm_output")
    if not parsed.in_report:
        return GateDecision(reason="llm_refused")
    text = parsed.answer.strip()
    if REFUSAL_WORDS.search(text):
        return GateDecision(reason="llm_refused")
    pages = {d.metadata["page"] for d in docs}
    context = "\n".join(d.page_content for d in docs)
    for year in re.findall(r"\b(?:19|20)\d{2}\b", question):
        if year not in context:
            return GateDecision(reason=f"year_{year}_not_in_evidence")
    evidence_numbers = numbers_in(context) + numbers_in(question)
    asked = numbers_in(question)
    for figure in numbers_in(text):
        printed, value, scale = figure
        if not figure_in_evidence(printed, value, scale, evidence_numbers):
            return GateDecision(reason=f"unsupported_figure_{printed}")
        if not figure_in_evidence(printed, value, scale, asked):
            placement = figure_placement(question, figure, docs)
            if placement != "ok":
                return GateDecision(reason=placement)
    cited = sorted(set(parsed.cited_pages) | {int(n) for n in CITATION.findall(text)})
    if any(page not in pages for page in cited):
        return GateDecision(reason="citation_outside_evidence")
    if not cited:  # cite the first passage printing the answer's figure, otherwise the top passage
        figures = [p for p, _, _ in numbers_in(text)]
        cited = [next((d for d in docs if any(f in d.page_content for f in figures)), docs[0]).metadata["page"]]
    if not CITATION.search(text):
        text = f"{text.rstrip('.')} " + " ".join(f"[p.{page}]" for page in cited) + "."
    return GateDecision(answer=text, cited_pages=cited)

def ground_truth_for(question, docs):
    """Used only when the LLM gives no answer. Never counted as an LLM answer in the metrics."""
    gold = GOLD_BY_QUESTION.get(question.strip().lower()) if GOLD_VALID else None
    if gold:
        return gold.ground_truth, "gold_set"
    if docs:
        doc = docs[0]
        return f"The model gave no verified answer. Closest passage, not a confirmed answer [p.{doc.metadata['page']}]: {doc.page_content[:500]}", "closest_passage"
    return "No verified information on this was found in the report.", "none"

def ask(question):
    started = time.time()
    docs = RETRIEVER.invoke(question)
    retrieved = time.time()
    raw, decision = "", GateDecision(reason="no_evidence")
    context = evidence_block(docs)
    if docs:
        raw = ANSWER_CHAIN.invoke({"context": context, "question": question})
        decision = gate_answer(question, raw, docs)
    if decision.answer:
        final, source, status = decision.answer, "llm", "LLM_ANSWER"
    else:
        final, source = ground_truth_for(question, docs)
        status = "GROUND_TRUTH_FALLBACK"
    return GroundedResult(question=question, status=status, final_answer=final, final_source=source, llm_answer=decision.answer,
                          llm_raw=raw, withheld_reason="" if decision.answer else decision.reason, cited_pages=decision.cited_pages,
                          retrieved_pages=[d.metadata["page"] for d in docs], context=context,
                          retrieval_s=round(retrieved - started, 3), total_s=round(time.time() - started, 3))

def show(result):
    print(f"Q: {result.question}\n{result.status} ({result.final_source}): {result.final_answer}")
    if result.withheld_reason:
        print(f"   LLM gave no answer: {result.withheld_reason}")
    print(f"   retrieved pages {result.retrieved_pages}, {result.total_s:.1f}s\n")

## Step 7: Try your own questions

In [ ]:
DEMO_QUESTIONS = [
    "What was Chevron's net income in 2024?",
    "In the five-year cumulative total returns graph, what was $100 invested in Chevron stock worth at the end of 2024?",
    "How much bitcoin does Chevron hold?",
]
for question in DEMO_QUESTIONS:
    show(ask(question))

## Step 8: Evaluate against the gold questions
- **Numeric number match:** the answer must contain the gold figure, exact or correctly rounded, with at most one other figure beside it.
- **Faithfulness:** the share of an answer's claims the evidence supports, in the style of RAGAS. The judge is Qwen in a separate call. A judge can be lenient and shares the model's blind spots, so treat this as an indicator rather than proof.
- **Negative refusal accuracy:** the share of unanswerable questions where the LLM gave no answer.
- **False refusals** are reported too, because refusing everything would otherwise score perfectly on the last two.

In [ ]:
class ClaimVerdict(BaseModel):
    supported: bool

JUDGE_PARSER = PydanticOutputParser(pydantic_object=ClaimVerdict)
JUDGE_CHAIN = ChatPromptTemplate.from_messages([
    ("system", 'You check a claim against evidence. It is supported only if the evidence directly states it, including the same figure, item and year. Reply with one JSON object and nothing else: {{"supported": true}} or {{"supported": false}}.'),
    ("human", "Evidence:\n{context}\n\nClaim: {claim}\nVerdict:"),
]) | CHAT_LLM | StrOutputParser()
FAITHFULNESS_TARGET = 0.98
REFUSAL_TARGET = 0.95
Rate = Annotated[float, Field(ge=0, le=1)]

class EvalRow(GroundedResult):
    id: str
    type: Literal["numeric", "text", "negative"]
    kind: str
    gold: str
    answered: bool
    correct: bool
    page_hit: Optional[bool] = None
    faithfulness: Optional[Rate] = None
    unsupported_claims: list[str] = Field(default_factory=list)

class ArchitectureMetrics(BaseModel):
    architecture: str
    questions: int = Field(gt=0)
    numeric_number_match_rate: Optional[Rate] = None
    number_match_tables: Optional[Rate] = None
    number_match_narrative: Optional[Rate] = None
    number_match_charts: Optional[Rate] = None
    text_fact_accuracy: Optional[Rate] = None
    faithfulness: Optional[Rate] = None
    negative_refusal_accuracy: Optional[Rate] = None
    false_refusal_rate: Optional[Rate] = None
    fallback_rate: Rate
    gold_page_retrieved: Optional[Rate] = None
    median_retrieval_s: float = Field(ge=0)
    median_total_s: float = Field(ge=0)

    @computed_field
    @property
    def faithfulness_pass(self) -> bool:
        return self.faithfulness is not None and self.faithfulness > FAITHFULNESS_TARGET

    @computed_field
    @property
    def negative_refusal_pass(self) -> bool:
        return self.negative_refusal_accuracy is not None and self.negative_refusal_accuracy > REFUSAL_TARGET

def number_match(result, gold):
    """Gold must appear and at most one other figure may sit beside it, so listing every column can't pass."""
    if not result.llm_answer:
        return False
    asked = {v for _, v, _ in numbers_in(gold.question)}
    target = gold.decimal
    figures = [f for f in numbers_in(result.llm_answer) if f[1] not in asked or f[1] == target]
    hit = any(same_figure(p, v, s, target, Decimal(gold.scale)) or same_figure(p, v, Decimal(1), target) for p, v, s in figures)
    return hit and len(figures) <= 2

ABBREVIATION = re.compile(r"(?:\b[A-Za-z]|\b(?:Dr|Mr|Mrs|Ms|Inc|Corp|Co|Ltd|No|St|vs|p\.l\.c))\.$")

def claims_of(answer):
    claims = []
    for piece in re.split(r"(?<=[.!?])\s+", CITATION.sub("", answer).strip()):
        if claims and ABBREVIATION.search(claims[-1]):
            claims[-1] = f"{claims[-1]} {piece}"
        else:
            claims.append(piece)
    return [c.strip() for c in claims if len(c.strip()) > 2]

def judge_claim(context, claim):
    """An unreadable verdict counts as unsupported."""
    try:
        return JUDGE_PARSER.parse(JUDGE_CHAIN.invoke({"context": context, "claim": claim})).supported
    except OutputParserException:
        return False

def faithfulness(result):
    """RAGAS-style: supported claims / claims. The judge is Qwen in a separate call; the gates never use it."""
    claims = claims_of(result.llm_answer)
    verdicts = [judge_claim(result.context, claim) for claim in claims]
    return (sum(verdicts) / len(verdicts) if verdicts else 0.0), [c for c, v in zip(claims, verdicts) if not v]

def run_evaluation():
    rows = []
    gold_rows = GOLD_SET[:CFG.eval_limit] if CFG.eval_limit else GOLD_SET
    for index, gold in enumerate(gold_rows, start=1):
        result = ask(gold.question)
        answered = bool(result.llm_answer)
        if gold.type == "numeric":
            correct = number_match(result, gold)
        elif gold.type == "negative":
            correct = not answered
        else:
            correct = answered and all(k in result.llm_answer.lower() for k in gold.keywords)
        score_value, unsupported = faithfulness(result) if answered else (None, [])
        rows.append(EvalRow(**result.model_dump(), id=gold.id, type=gold.type, kind=gold.kind, gold=gold.ground_truth, answered=answered, correct=correct,
                            page_hit=bool(set(gold.pages) & set(result.retrieved_pages)) if gold.pages else None,
                            faithfulness=score_value, unsupported_claims=unsupported))
        print(f"{index:>2}/{len(gold_rows)} {gold.id} {'ok ' if correct else 'MISS'} {result.status}: {(result.llm_answer or result.withheld_reason)[:90]}")
    return rows

def score(rows):
    frame = pd.DataFrame([row.model_dump() for row in rows])
    numeric, negative, text = (frame[frame.type == t] for t in ("numeric", "negative", "text"))
    answerable = frame[frame.type != "negative"]
    answered = frame[frame.answered]
    def rate(series):
        return float(series.mean()) if len(series) else None
    return frame, ArchitectureMetrics(
        architecture=ARCHITECTURE,
        questions=len(frame),
        numeric_number_match_rate=rate(numeric.correct),
        number_match_tables=rate(numeric[numeric.kind == "table"].correct),
        number_match_narrative=rate(numeric[numeric.kind == "narrative"].correct),
        number_match_charts=rate(numeric[numeric.kind == "chart"].correct),
        text_fact_accuracy=rate(text.correct),
        faithfulness=rate(answered.faithfulness.dropna()),
        negative_refusal_accuracy=rate(negative.correct),
        false_refusal_rate=rate(~answerable.answered),
        fallback_rate=rate(frame.status == "GROUND_TRUTH_FALLBACK"),
        gold_page_retrieved=rate(answerable.page_hit.dropna().astype(bool)),
        median_retrieval_s=float(frame.retrieval_s.median()),
        median_total_s=float(frame.total_s.median()))

if not GOLD_VALID:
    print("Skipped: the gold questions only apply to", EXPECTED_PDF)
else:
    results, metrics = score(run_evaluation())
    stem = CFG.work_dir / f"results_{ARCHITECTURE}"
    results.drop(columns=["context"]).to_csv(f"{stem}.csv", index=False)
    Path(f"{stem}.json").write_text(metrics.model_dump_json(indent=2))
    board = CFG.work_dir / "architecture_scoreboard.csv"
    previous = pd.read_csv(board) if board.exists() else pd.DataFrame()
    previous = previous[previous.architecture != ARCHITECTURE] if len(previous) else previous
    pd.concat([previous, pd.DataFrame([metrics.model_dump()])]).sort_values("architecture").to_csv(board, index=False)

    def verdict(value, passed):
        return "not measured" if value is None else "PASS" if passed else "FAIL"
    def pct(x):
        return "n/a" if x is None else f"{x:.1%}"
    print(f"\n=== {ARCHITECTURE}: {metrics.questions} questions, scored on the LLM's own answers only ===")
    print(f"Numeric number match rate : {pct(metrics.numeric_number_match_rate)} (tables {pct(metrics.number_match_tables)}, narrative {pct(metrics.number_match_narrative)}, charts {pct(metrics.number_match_charts)})")
    print(f"Faithfulness              : {pct(metrics.faithfulness)}  target > {FAITHFULNESS_TARGET:.0%}  {verdict(metrics.faithfulness, metrics.faithfulness_pass)}")
    print(f"Negative refusal accuracy : {pct(metrics.negative_refusal_accuracy)}  target > {REFUSAL_TARGET:.0%}  {verdict(metrics.negative_refusal_accuracy, metrics.negative_refusal_pass)}")
    print(f"Text fact accuracy        : {pct(metrics.text_fact_accuracy)}")
    print(f"False refusals            : {pct(metrics.false_refusal_rate)} of answerable questions (ground truth shown instead)")
    print(f"Gold page retrieved       : {pct(metrics.gold_page_retrieved)}")
    print(f"Median time               : {metrics.median_retrieval_s:.2f}s retrieval, {metrics.median_total_s:.1f}s total")
    misses = results[~results.correct][["id", "kind", "status", "llm_raw", "withheld_reason", "final_answer", "retrieved_pages"]]
    print(f"\n{len(misses)} misses:")
    with pd.option_context("display.max_colwidth", 90, "display.width", 220, "display.max_rows", None):
        print(misses.to_string(index=False))
    checked = results[results.answered & (results.type == "numeric")]
    if len(checked):
        wrong_passed = int(((checked.faithfulness == 1) & ~checked.correct).sum())
        right_failed = int(((checked.faithfulness < 1) & checked.correct).sum())
        print(f"Judge against gold figures : passed {wrong_passed} wrong answers, failed {right_failed} right answers, of {len(checked)}")
    unsupported = results[results.unsupported_claims.map(len) > 0][["id", "unsupported_claims"]]
    if len(unsupported):
        print("\nClaims the faithfulness judge did not support:")
        print(unsupported.to_string(index=False))

## Step 9: Download the results

In [ ]:
try:
    from google.colab import files
    for path in [CFG.work_dir / f"results_{ARCHITECTURE}.csv", CFG.work_dir / f"results_{ARCHITECTURE}.json"]:
        if path.exists():
            files.download(str(path))
except ImportError:
    print("Results are in", CFG.work_dir)